# **Зонд истинности, directional ablation и CCS: Практика**

В уроке «Отказ как одно направление» направление считалось разностью средних, а вмешательство убирало его из активаций. В уроке «Внутренние детекторы лжи и правдивость» те же инструменты читали истинность утверждений: зонд с учителем по Marks и Tegmark и CCS без меток по Burns et al. Там все считалось на синтетических активациях. Здесь те же три инструмента применяются к настоящей чат-модели Qwen2.5-0.5B-Instruct: 24 слоя, остаточный поток (residual stream) размерности 896.

В этой практике вы:

* соберете набор простых утверждений «Город X находится в стране Y», верных и неверных, и активации модели на последнем токене каждого утверждения на всех слоях;
* построите зонд истинности разностью средних (mass-mean probe, [Marks и Tegmark, 2024](https://arxiv.org/abs/2310.06824)) и найдете, с какого слоя он отделяет верные утверждения от неверных;
* уберете найденное направление из активаций операцией directional ablation, как в [Arditi et al., 2024](https://arxiv.org/abs/2406.11717), и проверите, остается ли истинность линейно читаемой;
* обучите CCS ([Burns et al., 2023](https://arxiv.org/abs/2212.03827)) на контрастных парах «Yes» / «No» без меток и сравните его точность с зондом с учителем.

Задания в тетради пронумерованы так же, как в уроке «Зонд истинности и CCS: Практика».

## **Подготовка окружения**
Хватит CPU: модель небольшая, и все вычисления здесь — прямые проходы и обучение линейных зондов. Прямые проходы по всем утверждениям занимают около минуты.

In [ ]:
!pip install transformers==5.18.0 -q

In [ ]:
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_grad_enabled(False)  # здесь только прямые проходы; градиенты включаются только для CCS
torch.manual_seed(0)
name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).eval()
print(model.config.num_hidden_layers, model.config.hidden_size)

## **Утверждения**

Для каждого из 69 городов возьмем верное утверждение «The city of X is in Y.» и неверное, где страна заменена случайной другой страной из того же списка. Треть городов уходит в тест целиком, вместе с обоими утверждениями: так зонд не увидит на тесте город, на котором учился.

In [ ]:
pairs = [("Paris", "France"), ("Berlin", "Germany"), ("Madrid", "Spain"), ("Rome", "Italy"), ("Lisbon", "Portugal"),
         ("Vienna", "Austria"), ("Warsaw", "Poland"), ("Prague", "the Czech Republic"), ("Budapest", "Hungary"), ("Athens", "Greece"),
         ("Stockholm", "Sweden"), ("Oslo", "Norway"), ("Helsinki", "Finland"), ("Copenhagen", "Denmark"), ("Dublin", "Ireland"),
         ("London", "the United Kingdom"), ("Amsterdam", "the Netherlands"), ("Brussels", "Belgium"), ("Bern", "Switzerland"), ("Moscow", "Russia"),
         ("Kyiv", "Ukraine"), ("Ankara", "Turkey"), ("Cairo", "Egypt"), ("Nairobi", "Kenya"), ("Lagos", "Nigeria"),
         ("Accra", "Ghana"), ("Tokyo", "Japan"), ("Beijing", "China"), ("Seoul", "South Korea"), ("Bangkok", "Thailand"),
         ("Hanoi", "Vietnam"), ("Jakarta", "Indonesia"), ("Manila", "the Philippines"), ("New Delhi", "India"), ("Karachi", "Pakistan"),
         ("Tehran", "Iran"), ("Baghdad", "Iraq"), ("Riyadh", "Saudi Arabia"), ("Ottawa", "Canada"), ("Toronto", "Canada"),
         ("Chicago", "the United States"), ("Boston", "the United States"), ("Mexico City", "Mexico"), ("Havana", "Cuba"), ("Lima", "Peru"),
         ("Bogota", "Colombia"), ("Santiago", "Chile"), ("Buenos Aires", "Argentina"), ("Montevideo", "Uruguay"), ("Caracas", "Venezuela"),
         ("Sydney", "Australia"), ("Melbourne", "Australia"), ("Auckland", "New Zealand"), ("Reykjavik", "Iceland"), ("Tallinn", "Estonia"),
         ("Riga", "Latvia"), ("Vilnius", "Lithuania"), ("Bucharest", "Romania"), ("Sofia", "Bulgaria"), ("Belgrade", "Serbia"),
         ("Zagreb", "Croatia"), ("Casablanca", "Morocco"), ("Tunis", "Tunisia"), ("Addis Ababa", "Ethiopia"), ("Kampala", "Uganda"),
         ("Mumbai", "India"), ("Osaka", "Japan"), ("Shanghai", "China"), ("Istanbul", "Turkey")]

rng = np.random.default_rng(0)
countries = sorted({k for _, k in pairs})
statements, labels, city = [], [], []
for i, (c, k) in enumerate(pairs):
    wrong = rng.choice([x for x in countries if x != k])
    statements += [f"The city of {c} is in {k}.", f"The city of {c} is in {wrong}."]
    labels += [1, 0]
    city += [i, i]
y = torch.tensor(labels)
test_cities = set(rng.permutation(len(pairs))[: len(pairs) // 3].tolist())
test = torch.tensor([c in test_cities for c in city])
train = ~test
print(len(statements), "утверждений;", int(test.sum()), "в тесте")
print(statements[:4])

## **Активации**

Прогоним каждое утверждение через модель и сохраним состояние остаточного потока на последнем токене (точке) на каждом слое: `hidden_states[0]` — эмбеддинги, `hidden_states[l]` — выход блока $l$.

In [ ]:
def last_token_states(texts):
    out = []
    for s in texts:
        ids = tok(s, return_tensors="pt")
        hs = model(**ids, output_hidden_states=True).hidden_states
        out.append(torch.stack([h[0, -1] for h in hs]))
    return torch.stack(out)  # [N, 25, 896]

H = last_token_states(statements)
print(H.shape)

## **Зонд разностью средних по слоям**

Зонд разностью средних устроен, как в коде Marks и Tegmark ([`geometry-of-truth`](https://github.com/saprmarks/geometry-of-truth): функция `collect_acts` с `center=True` и класс `MMProbe` в `probes.py`). Активации центрируются средним по всем утверждениям, направление $r = \bar h_{\text{true}} - \bar h_{\text{false}}$ считается по обучающим утверждениям, и утверждение считается верным, если проекция центрированной активации на $r$ больше нуля: это порог $\sigma(\theta_{mm}^\top h) > 0.5$ из статьи. Посчитаем точность такого зонда на тесте для каждого слоя.

In [ ]:
def mass_mean(X):
    Xc = X - X.mean(0)  # центрирование по всем утверждениям, как collect_acts(center=True) у Marks и Tegmark
    r = ...  # Ваш код здесь
    pred = ...  # Ваш код здесь
    return r, (pred[test] == y[test]).float().mean().item()

acc = [mass_mean(H[:, l])[1] for l in range(H.shape[1])]
for l, a in enumerate(acc):
    print(l, round(a, 3))

**Задание 1. Начиная с какого слоя точность зонда разностью средних на тесте не ниже 0.9? Запишите номер слоя.**

**Задание 2. Чему равна точность зонда разностью средних на тесте на слое 16? Ответ округлите до сотых.**

## **Directional ablation направления истинности**

Возьмем слой 16. Directional ablation убирает из каждой активации компоненту вдоль единичного направления $\hat r = r/\lVert r\rVert$:

$$h \leftarrow h - (\hat r^\top h)\,\hat r.$$

После этого зонд разностью средних, посчитанный на тех же активациях, бесполезен по построению: разность средних после абляции — нулевой вектор. Вопрос другой: остается ли истинность линейно читаемой по другим направлениям? Проверим логистической регрессией (признаки стандартизованы по обучающим утверждениям), обученной на активациях до и после абляции. Активации берем в float64, чтобы проекция на $\hat r$ после абляции была нулем с точностью до округления.

У Arditi et al. абляция идет во время прямого прохода, на всех слоях и всех позициях токенов, и меняет ответ модели. Здесь она применяется к сохраненным активациям одного слоя: вопрос не о поведении модели, а о том, что остается линейно читаемым на этом слое.

In [ ]:
L = 16
X = H[:, L].double()
r, _ = mass_mean(X)
r_hat = r / r.norm()
X_abl = ...  # Ваш код здесь

def logistic_acc(Z):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000))
    clf.fit(Z[train].numpy(), y[train].numpy())
    return clf.score(Z[test].numpy(), y[test].numpy())

print("max |проекция| после абляции:", (X_abl @ r_hat).abs().max().item())
print("логистическая регрессия до абляции:   ", round(logistic_acc(X), 3))
print("логистическая регрессия после абляции:", round(logistic_acc(X_abl), 3))

**Задание 3. Чему равна точность логистического зонда на тесте на слое 16 после directional ablation направления $\hat r$? Ответ округлите до сотых.**

**Задание 4. Что показывает результат задания 3?**

## **CCS: истинность без меток**

CCS строит для каждого утверждения контрастную пару: «Is it true that the city of X is in Y? Yes» и тот же вопрос с ответом «No». Активации каждой половины пары центрируются отдельно, как в функции `normalize` кода авторов ([`discovering_latent_knowledge`](https://github.com/collin-burns/discovering_latent_knowledge), `utils.py`): из обучающих утверждений вычитается среднее по обучающим, из тестовых — среднее по тестовым. Так зонд не может читать сам токен «Yes» или «No». Зонд $p(h) = \sigma(w^\top h + b)$ обучается без меток, на лоссе

$$L = \big(p^+ - (1 - p^-)\big)^2 + \min(p^+, p^-)^2,$$

где первый член требует согласованности ($p^+ \approx 1 - p^-$), а второй запрещает вырожденный ответ $p^+ = p^- = 0.5$. Знак направления CCS не определяет, поэтому точность берут как $\max(\text{acc}, 1 - \text{acc})$. Как в коде авторов, обучим десять зондов из разных случайных начал (AdamW, скорость обучения $10^{-3}$, weight decay 0.01, 1000 эпох на всех обучающих парах) и оставим тот, у которого лосс на обучающих парах меньше.

*Статья и код авторов здесь расходятся. В статье (раздел 2.2 и приложение G) после центрирования активации делят еще и на их среднюю норму, а скорость обучения равна 0.01; в коде по умолчанию масштаб не нормируется (`var_normalize=False`), а скорость обучения равна $10^{-3}$. Тетрадь идет за кодом.*

In [ ]:
questions = [s[:-1].replace("The city of", "Is it true that the city of") + "?" for s in statements]
Hp = last_token_states([q + " Yes" for q in questions])[:, L]
Hn = last_token_states([q + " No" for q in questions])[:, L]

def normalize(A, split):  # каждая половина пары и каждая часть выборки центрируются своим средним
    return A[split] - A[split].mean(0)

Ptr, Ntr, Pte, Nte = normalize(Hp, train), normalize(Hn, train), normalize(Hp, test), normalize(Hn, test)

def ccs_loss(w, b, P, N):
    pp, pn = torch.sigmoid(P @ w + b), torch.sigmoid(N @ w + b)
    return ...  # Ваш код здесь

best = None
with torch.enable_grad():
    for seed in range(10):
        torch.manual_seed(seed)
        lin = torch.nn.Linear(Hp.shape[1], 1)  # инициализация nn.Linear, как в коде авторов
        w, b = lin.weight[0], lin.bias
        opt = torch.optim.AdamW(lin.parameters(), lr=1e-3, weight_decay=0.01)
        for _ in range(1000):
            loss = ccs_loss(w, b, Ptr, Ntr)
            opt.zero_grad(); loss.backward(); opt.step()
        if best is None or loss.item() < best[0]:
            best = (loss.item(), w.detach(), b.detach(), seed)

loss, w, b, seed = best
score = 0.5 * (torch.sigmoid(Pte @ w + b) + 1 - torch.sigmoid(Nte @ w + b))
a = ((score > 0.5).long() == y[test]).float().mean().item()
print("сид", seed, "функция потерь", round(loss, 4))
print("точность CCS:", round(max(a, 1 - a), 3))

**Задание 5. Чему равна точность CCS на тесте на слое 16? Ответ округлите до сотых.**

## **Заключение**

- Истинность простых утверждений о городах в Qwen2.5-0.5B-Instruct читается линейно: зонд разностью средних, обученный на двух третях городов, отделяет верные утверждения от неверных на остальных, но только начиная со слоя 8 из 24: на первых четырех слоях его точность около 0.5, на слоях 4–7 — от 0.61 до 0.67.
- Directional ablation одного направления $\hat r$ — та же операция, что убирает направление отказа у Arditi et al. Когда $\hat r$ — разность средних классов, после абляции средние классов совпадают, и ни один линейный классификатор не отличает классы лучше константы: это условие линейного стирания понятия из работы [Belrose et al., 2023](https://arxiv.org/abs/2306.03819) (LEACE). Поэтому логистический зонд после абляции падает до 0.5, хотя до нее он был точнее зонда разностью средних. Нелинейный зонд или сама модель на следующих слоях могут читать истинность и дальше: абляция на одном слое стирает только линейно читаемую ее часть.
- CCS находит направление истинности без меток: на слое 16 его точность 0.957 — та же, что у зонда разностью средних, и ниже, чем у логистического зонда (1.0); оба зонда обучены с метками. Но низкий лосс CCS сам по себе не доказывает, что найдена истинность, — об этом возражения Farquhar et al. (2023) из урока «Внутренние детекторы лжи и правдивость»: здесь точность удалось проверить только потому, что у утверждений есть метки. Поменяйте `L` на 8 или 12 и посмотрите, что станет с лоссом и точностью CCS.
- Набор здесь маленький: 46 тестовых утверждений, шаг точности $1/46 \approx 0.022$. Поэтому сравнивайте точности с этой поправкой.